# Setup — Semantic Layer & Genie Lab

Run this once, top to bottom. About 3 minutes, no code required from you. It is safe to re-run:
every object is created with `CREATE OR REPLACE` / overwrite.

It creates, all inside one schema, `<you>_semantic`:

1. **Reference tables:** 90-day revenue targets per region and the company's written metric glossary.
2. **Synthetic retail data:** 180 days of orders, refunds, customers and products for *Lumen Outfitters*, an outdoor-gear web shop, plus one legacy summary table.
3. **Lab views:** what every exercise, metric view and Genie Agent queries.

**Compute:** serverless. Exercises 2–4 also need a **pro or serverless SQL warehouse** you can use (Genie runs its SQL there).

In [ ]:
#Run this notebook with a serverless instance

from helpers import utils, setup_schemas, reference_data, shop_data, lab_views

## 1 — Your namespace

The catalog is set once, in `helpers/utils.py` (`DEFAULT_CATALOG`). Your schema name comes from
`current_user()`: `jane.doe@factored.ai` gets `jane_doe_semantic`.

In [ ]:
config = utils.get_configs()
for key in ("catalog", "base_user", "schema", "metric_view"):
    print(f"{key:12s} {config[key]}")

## 2 — Schema

If your instructor already created your schema, it is reused and you need no `CREATE SCHEMA`
privilege. If creation fails, the error message lists the exact `GRANT` your instructor has to run.

In [ ]:
setup_schemas.create_user_schema()

## 3 — Reference tables

| Table | Purpose |
|---|---|
| `region_targets` | 90-day **net** revenue target per region (Exercise 2). |
| `metric_glossary` | How Finance and RevOps define revenue and active customers. Nobody reads it; that is the problem (Exercise 1). |

In [ ]:
reference_data.create_reference_tables()

## 4 — Synthetic retail data

Fresh training workspaces have no business data, so the lab ships its own. It is seeded, so everyone
gets the same picture, and dates are relative to today (the last day of data is yesterday).

**Two everyday metrics are quietly ambiguous in this data.** Finding out why is Exercise 1.

In [ ]:
summary = shop_data.write_synthetic_tables()

## 5 — Lab views

| View | Source | Grain |
|---|---|---|
| `lab_customers` | `synthetic_customers` | one row per customer |
| `lab_products` | `synthetic_products` | one row per product |
| `lab_orders` | `synthetic_orders` | one row per order |
| `lab_refunds` | `synthetic_refunds` | one row per **refund event** (an order can have several) |
| `lab_daily_sales_summary` | `synthetic_daily_sales_summary` | one row per day (legacy) |
| `lab_region_targets` | `region_targets` | one row per region |

Exercises, the metric view and both Genie Agents point at these views, never at the
`synthetic_*` tables. To run the lab on real data later, re-point the views and nothing else changes.

In [ ]:
lab_views.bind_views()

## 6 — Verify

In [ ]:
%sql
-- Expect ~180 days, ~40k orders, a few hundred test orders and ~4.5% cancelled
SELECT
  MIN(order_date)                         AS first_day,
  MAX(order_date)                         AS last_day,
  COUNT(*)                                AS orders,
  COUNT(DISTINCT customer_id)             AS customers,
  COUNT_IF(is_test_order)                 AS test_orders,
  ROUND(AVG(CAST(status = 'cancelled' AS INT)) * 100, 1) AS cancelled_pct,
  (SELECT COUNT(*) FROM lab_refunds)      AS refund_events
FROM lab_orders

In [ ]:
cfg = utils.get_configs()
print(f"Namespace ready: {cfg['catalog']}.{cfg['schema']}\n")
for name in utils.SYNTHETIC_TABLES + utils.REFERENCE_TABLES + utils.LAB_VIEWS:
    fqn = utils.fq(name)
    status = f"{utils.get_table_row_count(fqn):>8,} rows" if utils.table_exists(fqn) else "MISSING"
    print(f"  {name:32s} {status}")

---

Setup is done. Work through the exercises in order:

`01-semantic-layer-and-metric-views` → `02-genie-agents` → `03-genie-vs-raw-tables` → `04-genie-ontology-and-genie-one`